In [3]:
from prefect import flow, task
import requests
import pandas as pd
from pathlib import Path

# new import
import pandera.pandas as pa
from pandera import Column, DataFrameSchema, Check

from common.loaders import load_valid_country_ids

API_URL = "https://restcountries.com/v3.1/all?fields=cca3,currencies"

# --- Esquema de validación ---
currencies_schema = pa.DataFrameSchema({
    "id_country": Column(str, [
        Check.str_length(3, 3),
        Check.isin(load_valid_country_ids())
    ], nullable=False),
    "id_currency": Column(str, [
        Check.str_length(3, 3)  # códigos de moneda ISO
    ], nullable=False),
    "currency_name": Column(str, nullable=True),
    "symbol": Column(str, nullable=True),
})

@task
def extract_currencies():
    response = requests.get(API_URL)
    response.raise_for_status()
    return response.json()

@task
def transform_currencies(data):
    df = pd.DataFrame(data)

    # Normalizar currencies
    df_currencies = (
        df[["cca3", "currencies"]]
        .assign(currencies=df["currencies"].apply(lambda x: x if isinstance(x, dict) else {}))
        .assign(currency_list=lambda d: d["currencies"].apply(lambda c: list(c.items())))
        .explode("currency_list")
        .dropna(subset=["currency_list"])
    )

    # Separar clave y valores
    df_currencies[["id_currency", "details"]] = pd.DataFrame(
        df_currencies["currency_list"].tolist(), index=df_currencies.index
    )

    # Expandir detalles (name y symbol)
    df_currencies["currency_name"] = df_currencies["details"].apply(
        lambda x: x.get("name") if isinstance(x, dict) else None
    )
    df_currencies["symbol"] = df_currencies["details"].apply(
        lambda x: x.get("symbol") if isinstance(x, dict) else None
    )

    # Renombrar y limpiar
    df_currencies = df_currencies.rename(columns={"cca3": "id_country"})[
        ["id_country", "id_currency", "currency_name", "symbol"]
    ]

    return df_currencies

@task
def validate_currencies(df: pd.DataFrame) -> pd.DataFrame:
    return currencies_schema.validate(df)

@task
def load_currencies(df: pd.DataFrame):
    file_path = Path("../stage/country_currencies.csv")
    file_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(file_path, index=False, encoding="utf-8")
    print(f"Saved {len(df)} rows to {file_path}")

@flow(name="etl_currencies_flow")
def etl_currencies():
    data = extract_currencies()
    df = transform_currencies(data)
    df = validate_currencies(df)  # validación con pandera
    load_currencies(df)

if __name__ == "__main__":
    etl_currencies()


03:42:53.015 | INFO    | Flow run 'bouncy-silkworm' - Beginning flow run 'bouncy-silkworm' for flow 'etl_currencies_flow'

03:42:54.125 | INFO    | Task run 'extract_currencies-fe2' - Finished in state Completed()

03:42:54.366 | INFO    | Task run 'transform_currencies-3ad' - Finished in state Completed()

03:42:54.608 | INFO    | Task run 'validate_currencies-f18' - Finished in state Completed()

Saved 268 rows to ..\stage\country_currencies.csv


03:42:54.828 | INFO    | Task run 'load_currencies-2e1' - Finished in state Completed()

03:42:54.839 | INFO    | Flow run 'bouncy-silkworm' - Finished in state Completed()